# TinyPFA Analysis Notebook June 2026

This notebook loads all `.txt` files matching a user-specified prefix, sorted by the timestamp in each filename. It then calculates and plots various diagnostics meant for debugging the Leo Bodnars but, in principle, any GNSS receiver we're interested in.

In [ ]:
%matplotlib widget

# Numerical work, plotting, MTIE/ADEV utilities, and path handling.
from pathlib import Path
import allantools
import matplotlib.pyplot as plt
import numpy as np
import re
import sys

In [ ]:
#Note that the first couple of lines from the tinyPFA can be corrupted. Right now, the fix is to go in and manually delete these. 

# The prefix identifies one measurement campaign. All matching .txt files are
# loaded and sorted by the timestamp embedded in the filename.
file_prefix = 'tinypfa_lb1420_splitter_nocleaner_test_10ms_unwrapped_6-02-26'
apply_unwrap = False  # Set True for wrapped phase files, False for already-unwrapped files
phase_scale_seconds = 1.0  # New unwrapped files are already seconds; old fraction-of-100-ns files used 100e-9
data_dir = Path('.')

# Sort files chronologically using the _YYYYMMDD_HHMMSS suffix.
# If a filename does not match that pattern, fall back to the filename itself.
def timestamp_key(path):
    match = re.search(r'_(\d{8})_(\d{6})', path.stem)
    if match is None:
        return path.name
    return ''.join(match.groups())

# Look first in the current working directory, then in a nested LB_Clock directory.
data_files = sorted(data_dir.glob(f'{file_prefix}*.txt'), key=timestamp_key)
if not data_files:
    alt_data_dir = Path('LB_Clock')
    data_files = sorted(alt_data_dir.glob(f'{file_prefix}*.txt'), key=timestamp_key)

if not data_files:
    raise FileNotFoundError(f'No files found matching prefix {file_prefix!r}')

# Read the first whitespace-separated value from each non-empty line.
# These files store one phase/TIE sample per line.
raw_values = []
for data_file in data_files:
    print(f'Loading {data_file}')
    with data_file.open() as f:
        for line in f:
            stripped = line.strip()
            if not stripped:
                continue
            raw_values.append(float(stripped.split()[0]))

# Convert input values to seconds
phase = np.array(raw_values) * phase_scale_seconds
rate = 100.0  # 10 ms cadence in Hz (set from the tinyPFA)

print(f'Loaded {len(phase)} samples from {len(data_files)} file(s)')
print(f'Phase scale: {phase_scale_seconds:g} seconds/input unit')

# Optional unwrap path for older wrapped phase files. The 100 ns period is the
# wrap period used by the older fraction-of-100-ns data format.
period = 100e-9 # 10 MHz
threshold = period * 0.5

if apply_unwrap:
    unwrapped_phase = phase.copy()
    delta = np.diff(unwrapped_phase)
    correction = np.where(delta > threshold, -period, np.where(delta < -threshold, period, 0.0))
    unwrapped_phase[1:] = unwrapped_phase[1:] + np.cumsum(correction)
else:
    unwrapped_phase = phase.copy()

print(f'Input phase range: {(phase.max() - phase.min()) * 1e9:.3f} ns')
print(f'Processed phase range: {(unwrapped_phase.max() - unwrapped_phase.min()) * 1e9:.3f} ns')
print(f'Unwrap applied: {apply_unwrap}')

# Skip initial settling time before analysis. This keeps startup transients from
# affecting the MTIE/TIE statistics.
startTime = 0  # seconds to skip at the start of the data
unwrapped_phase = np.asarray(unwrapped_phase[int(rate*startTime):])

# Diagnostic: standard deviation of 1-second averaged phase differences.
# This is a quick time-domain jitter check, not the MTIE calculation itself.
diffs = np.std(np.diff(unwrapped_phase[:int((len(unwrapped_phase)//rate)*rate)].reshape(-1, int(rate)).mean(axis=1)),ddof=1)



# MTIE can be slow for long records, so cache the octave-tau result.
# The filename starts with file_prefix and includes startTime so different trims
# do not accidentally reuse the same cached arrays.

#Note that if you're still collecting data, you need to delete this cached file and re-compute. 
mtie_cache_file = Path(f'{file_prefix}_start{startTime:g}s_mtie_octave.npz')
if mtie_cache_file.exists():
    cached_mtie = np.load(mtie_cache_file)
    taus = cached_mtie['taus']
    mtie = cached_mtie['mtie']
    mtie_err = cached_mtie['mtie_err']
    ns = cached_mtie['ns']
    print(f'Loaded cached MTIE data from {mtie_cache_file}')
else:
    taus, mtie, mtie_err, ns = allantools.mtie(unwrapped_phase, rate=rate, data_type='phase', taus='octave')
    np.savez(mtie_cache_file, taus=taus, mtie=mtie, mtie_err=mtie_err, ns=ns)
    print(f'Saved MTIE data to {mtie_cache_file}')

In [ ]:
plt.close('all')

# Print the cached/computed MTIE table in ns so the numeric values are visible
# even before inspecting the plot.
for val in zip(taus, mtie):
	print(f'{val[0]:.3e} \t\t {val[1]*1e9:.3f}')
#print(taus)
#Zprint(mtie)
# Formatting for plots
label_font = {"fontname": "sans-serif", "size": "16", "color": "black", "weight": "bold", "verticalalignment": "bottom"}
title_font = {"fontname": "sans-serif", "size": "16", "color": "black", "weight": "bold"}
legend_font = {"family": "sans-serif", "size": "12", "style": "normal", "weight": "bold"}

# Plot MTIE versus tau. The MTIE arrays are in seconds, so multiply by 1e9 to
# show nanoseconds on the y-axis.
fig, ax = plt.subplots(figsize=(8, 6))
ax.errorbar(taus, mtie * 1e9, yerr=mtie_err * 1e9, fmt='o-', capsize=3, label='tinypfa LB test', alpha=0.8)

ax.set_xscale('log')
ax.set_xlim(taus[0] / 1.3, taus[-1] * 1.5)
ax.set_xlabel('Time Interval τ (s)', labelpad=15, fontdict=label_font)
ax.set_ylabel('MTIE (ns)', fontdict=label_font)
ax.tick_params(axis='both', labelsize=12)
for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
    tick_label.set_fontweight('bold')
ax.grid(True, which='both', linestyle='--', alpha=0.5)
#ax.legend(prop=legend_font)
ax.margins(y=0.1)
fig.subplots_adjust(left=0.16, right=0.96, bottom=0.18, top=0.95)

In [ ]:
plt.close('all')

# Formatting for plots
label_font = {"fontname": "sans-serif", "size": "16", "color": "black", "weight": "bold", "verticalalignment": "bottom"}
title_font = {"fontname": "sans-serif", "size": "16", "color": "black", "weight": "bold"}
legend_font = {"family": "sans-serif", "size": "12", "style": "normal", "weight": "bold"}


# Convert phase to a TIE trace in ns, referenced to the first retained sample.
# elapsed_time is in seconds and has the same length as unwrapped_phase.
elapsed_time = np.arange(len(unwrapped_phase)) / rate
tie_ns = (unwrapped_phase - unwrapped_phase[0]) * 1e9

# Fit a straight line to the TIE. The slope is in ns/s; multiplied by 1e-9 it
# is the corresponding fractional frequency offset.
tie_slope_ns_per_s, tie_intercept_ns = np.polyfit(elapsed_time, tie_ns, 1)
tie_fit_ns = tie_slope_ns_per_s * elapsed_time + tie_intercept_ns
fractional_frequency_error = tie_slope_ns_per_s * 1e-9

# R² measures how much TIE variance is explained by a linear trend. It is shown
# on the plot, but the slope confidence interval below is the better test for
# whether the slope is statistically consistent with zero.
r_value = np.corrcoef(elapsed_time, tie_ns)[0, 1]
r_squared = r_value ** 2

print(f'TIE linear fit slope: {tie_slope_ns_per_s:.6g} ns/s')
print(f'Fractional frequency error: {fractional_frequency_error:.6g}')
print(f'R value: {r_value:.6g}')
print(f'R² value: {r_squared:.6g}')

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(elapsed_time, tie_ns, linewidth=0.8, alpha=0.85, label='tinypfa LBE-1420 test')
ax.plot(elapsed_time, tie_fit_ns, '--', linewidth=2.0,
        label=f'Linear fit: {tie_slope_ns_per_s:.3g} ns/s, R²={r_squared:.3f}')

ax.set_title('Time Interval Error', fontdict=title_font)
ax.set_xlabel('Elapsed Time (s)', labelpad=15, fontdict=label_font)
ax.set_ylabel('Time Interval Error (ns)', fontdict=label_font)
ax.tick_params(axis='both', labelsize=12)
for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
    tick_label.set_fontweight('bold')
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.legend(prop=legend_font)
ax.margins(x=0.02, y=0.1)
fig.subplots_adjust(left=0.18, right=0.96, bottom=0.18, top=0.92)

In [ ]:
# Study autocorrelation in the linear-fit residuals, not the raw TIE.
# The raw TIE includes the trend we just fit, which would artificially lengthen
# the apparent correlation time.
tie_residual_ns = tie_ns - tie_fit_ns
tie_residual_ns = tie_residual_ns - np.mean(tie_residual_ns)

# Normalized residual autocorrelation using an FFT. The zero padding avoids
# circular wraparound, and the normalization makes ACF[0] = 1.
n_resid = len(tie_residual_ns)
fft_len = 2 ** int(np.ceil(np.log2(2 * n_resid - 1)))
resid_fft = np.fft.fft(tie_residual_ns, fft_len)
resid_acf = np.fft.ifft(resid_fft * np.conjugate(resid_fft)).real[:n_resid]
resid_acf /= resid_acf[0]

acf_lags_s = np.arange(n_resid) / rate

# Two simple ACF-derived correlation scales:
#   tau_1e_s: characteristic decay time where ACF first falls below 1/e.
#   tau_zero_s: end of the first positive-correlation lobe.
# These are diagnostics for choosing HAC/Newey-West maxlags.
below_1e = np.where(resid_acf < 1 / np.e)[0]
tau_1e_s = acf_lags_s[below_1e[0]] if len(below_1e) else np.nan

zero_crossings = np.where(resid_acf < 0)[0]
tau_zero_s = acf_lags_s[zero_crossings[0]] if len(zero_crossings) else np.nan

# Prefer the first zero crossing as a conservative default when it exists;
# otherwise fall back to the 1/e time. The HAC cell still sweeps several values
# because the conclusion can depend on this choice.
if np.isfinite(tau_zero_s):
    suggested_hac_maxlags = int(np.ceil(tau_zero_s * rate))
    suggested_from = 'first zero crossing'
elif np.isfinite(tau_1e_s):
    suggested_hac_maxlags = int(np.ceil(tau_1e_s * rate))
    suggested_from = '1/e time'
else:
    suggested_hac_maxlags = None
    suggested_from = None

print(f'Residual ACF 1/e time: {tau_1e_s:.6g} s')
print(f'Residual ACF first zero crossing: {tau_zero_s:.6g} s')
if suggested_hac_maxlags is not None:
    print(f'Suggested HAC maxlags from {suggested_from}: {suggested_hac_maxlags} samples ({suggested_hac_maxlags / rate:.6g} s)')
else:
    print('No finite ACF-based HAC maxlags suggestion found.')

# Limit the displayed x range for readability. The ACF search above still used
# the full computed ACF, not just the plotted window.
acf_plot_max_s = min(10000, acf_lags_s[-1])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(acf_lags_s, resid_acf, linewidth=1.0, label='TIE fit residual ACF')
ax.axhline(0, color='black', linewidth=1)
ax.axhline(1 / np.e, color='gray', linestyle='--', linewidth=1.5, label='1/e')
if np.isfinite(tau_1e_s):
    ax.axvline(tau_1e_s, color='tab:orange', linestyle='--', linewidth=1.5, label=f'1/e: {tau_1e_s:.3g} s')
if np.isfinite(tau_zero_s):
    ax.axvline(tau_zero_s, color='tab:red', linestyle='--', linewidth=1.5, label=f'zero: {tau_zero_s:.3g} s')
ax.set_xlim(0, acf_plot_max_s)
ax.set_xlabel('Lag (s)', labelpad=15, fontdict=label_font)
ax.set_ylabel('Residual autocorrelation', fontdict=label_font)
ax.set_title('TIE Linear-Fit Residual Autocorrelation', fontdict=title_font)
ax.tick_params(axis='both', labelsize=12)
for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
    tick_label.set_fontweight('bold')
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.legend(prop=legend_font)
fig.subplots_adjust(left=0.18, right=0.96, bottom=0.18, top=0.90)

In [ ]:
import statsmodels.api as sm

# Test whether the TIE slope is statistically consistent with zero.
# x is elapsed time in seconds; y is TIE in ns.
x = elapsed_time
y = tie_ns

# Skipping this cell by default since it can take a long time to run, and the ACF diagnostics above 
if False:
	# Try a range of correlation windows in seconds. HAC/Newey-West expands the
	# slope uncertainty to account for time-correlated residuals. Heads-up that
	# this can take forever, so you may want to skip this cell
	maxlags = [0.1, 1, 10, 100, 1000, 2000, 3000]

	# statsmodels expects maxlags as a number of samples, so each auto_lag is
	# converted from seconds to samples with int(auto_lag * rate).
	X = sm.add_constant(x)

	for auto_lag in maxlags:
		print(f'\nTesting with maxlags = {auto_lag}')
		model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": int(auto_lag*rate)})

		intercept, slope = model.params
		alpha = 0.01
		slope_ci = model.conf_int(alpha=alpha)[1]

		print(f"Slope = {slope:.6e} ns/s")
		print(f"{100*(1-alpha):.6f}% HAC CI = [{slope_ci[0]:.6e}, {slope_ci[1]:.6e}] ns/s")

		if slope_ci[0] <= 0 <= slope_ci[1]:
			print(f"Slope is consistent with zero at {100*(1-alpha):.6f}% confidence.")
		else:
			print(f"Slope is not consistent with zero at {100*(1-alpha):.6f}% confidence.")
		print('---')

In [ ]:
plt.close('all')

# Formatting for plots
label_font = {"fontname": "sans-serif", "size": "16", "color": "black", "weight": "bold", "verticalalignment": "bottom"}
title_font = {"fontname": "sans-serif", "size": "16", "color": "black", "weight": "bold"}
legend_font = {"family": "sans-serif", "size": "12", "style": "normal", "weight": "bold"}

f0 = 1e7
rate = 100
elapsed_time = (np.arange(len(unwrapped_phase)) / rate)[:-1]
tie_ns = (unwrapped_phase - unwrapped_phase[0]) * 1e9
freq_err = f0*(tie_ns[1:] - tie_ns[:-1])*rate/1e9


fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(elapsed_time, freq_err, linewidth=0.8, alpha=0.85, label='tinypfa LBE-1420 test')

ax.set_title('Frequency Error', fontdict=title_font)
ax.set_xlabel('Elapsed Time (s)', labelpad=15, fontdict=label_font)
ax.set_ylabel('Frequency Error (Hz)', fontdict=label_font)
ax.tick_params(axis='both', labelsize=12)
for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
    tick_label.set_fontweight('bold')
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.legend(prop=legend_font)
ax.margins(x=0.02, y=0.1)
fig.subplots_adjust(left=0.18, right=0.96, bottom=0.18, top=0.92)

In [ ]:
plt.close('all')

# Formatting for plots
label_font = {"fontname": "sans-serif", "size": "16", "color": "black", "weight": "bold", "verticalalignment": "bottom"}
title_font = {"fontname": "sans-serif", "size": "16", "color": "black", "weight": "bold"}
legend_font = {"family": "sans-serif", "size": "12", "style": "normal", "weight": "bold"}

elapsed_time = np.arange(len(unwrapped_phase)) / rate
tie_ns = (unwrapped_phase - unwrapped_phase[0]) * 1e9
tie_slope_ns_per_s, tie_intercept_ns = np.polyfit(elapsed_time, tie_ns, 1)
tie_fit_ns = tie_slope_ns_per_s * elapsed_time + tie_intercept_ns
tie_residual_ns = tie_ns - tie_fit_ns

print(f'Removed TIE slope: {tie_slope_ns_per_s:.6g} ns/s')
print(f'Residual TIE range: {np.ptp(tie_residual_ns):.6g} ns')

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(elapsed_time, tie_residual_ns, linewidth=0.8, alpha=0.85, label='Detrended tinypfa LB test')
ax.axhline(0, color='black', linewidth=1.0, alpha=0.7)

ax.set_title('Detrended Time Interval Error', fontdict=title_font)
ax.set_xlabel('Elapsed Time (s)', labelpad=15, fontdict=label_font)
ax.set_ylabel('Residual TIE (ns)', fontdict=label_font)
ax.tick_params(axis='both', labelsize=12)
for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
    tick_label.set_fontweight('bold')
ax.grid(True, which='both', linestyle='--', alpha=0.5)
# ax.legend(prop=legend_font)
ax.margins(x=0.02, y=0.1)
fig.subplots_adjust(left=0.18, right=0.96, bottom=0.18, top=0.92)

In [ ]:
plt.close('all')

# Compute Allan deviation directly from the TIE time-error series.
# TIE is in nanoseconds, so the second-difference formula below is converted
# back to fractional frequency by dividing by 1e9.
tie_ns = (unwrapped_phase - unwrapped_phase[0]) * 1e9

tau0 = 0.01  # 10 ms sample interval
n = len(tie_ns)
# Use octave-spaced averaging factors so the plot spans the full record without
# producing an overly dense set of tau values.
max_m = n // 2
max_power = int(np.floor(np.log2(max_m)))
m_values = 2 ** np.arange(max_power + 1)
taus_adev = m_values * tau0

# Allan deviation from phase/TIE uses the second difference x[t+2tau] - 2x[t+tau] + x[t].
adev = []
for m in m_values:
    diff = tie_ns[2*m:] - 2 * tie_ns[m:-m] + tie_ns[:-2*m]
    variance = np.sum(diff * diff) / (2 * (m * tau0) ** 2 * len(diff))
    adev.append(np.sqrt(variance) / 1e9)  # convert ns/s to fractional frequency
adev = np.array(adev)

print('Allan deviation from TIE (fractional):')
for tau, a in zip(taus_adev, adev):
    print(f'τ = {tau:.3f} s, ADEV = {a:.3e}')

label_font = {"fontname": "sans-serif", "size": 16, "color": "black", "weight": "bold", "verticalalignment": "bottom"}
title_font = {"fontname": "sans-serif", "size": 16, "color": "black", "weight": "bold"}
legend_font = {"family": "sans-serif", "size": 12, "style": "normal", "weight": "bold"}

fig, ax = plt.subplots(figsize=(8, 6))
ax.loglog(taus_adev, adev, 'o-', label='ADEV from TIE', alpha=0.85)
ax.set_xlabel('Time Interval τ (s)', labelpad=15, fontdict=label_font)
ax.set_ylabel('Allan deviation', fontdict=label_font)
ax.set_title('Allan Deviation from TIE', fontdict=title_font)
ax.grid(True, which='both', linestyle='--', alpha=0.5)
for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
    tick_label.set_fontweight('bold')
ax.legend(prop=legend_font)
fig.subplots_adjust(left=0.16, right=0.96, bottom=0.18, top=0.92)


In [ ]:
# This cell estimates the fractional deviation implied by TIE differences over
# each tau and compares it to a 1 ns / (sqrt(2) * tau) requirement line.
# TIE is in nanoseconds.
tie_ns = (unwrapped_phase - unwrapped_phase[0]) * 1e9

tau0 = 0.01  # 10 ms sample interval
n = len(tie_ns)

# Keep tau <= half the record length so every difference uses at least one pair
# of samples, then use octave spacing for readability.
max_m = n // 2
max_power = int(np.floor(np.log2(max_m)))
m_values = 2 ** np.arange(max_power + 1)
taus = m_values * tau0

frac_std = []
frac_rmse = []

for m in m_values:
    tau = m * tau0

    # TIE difference over interval tau, in ns
    diff_ns = tie_ns[m:] - tie_ns[:-m]

    # Convert to fractional frequency-like quantity: seconds / seconds
    y_tau = (diff_ns * 1e-9) / tau

    # Random variation, mean removed
    frac_std.append(np.std(y_tau, ddof=1))

    # RMSE, includes mean frequency offset
    frac_rmse.append(np.sqrt(np.mean(y_tau**2)))

frac_std = np.array(frac_std)
frac_rmse = np.array(frac_rmse)

taus_req = np.concatenate([[0.8*taus[0]], taus, [1.2*taus[-1]]])
requirement = 1e-9 / taus_req / np.sqrt(2)
# requirement is evaluated on an extended tau grid for smooth-looking shading;
# requirement_at_data is evaluated at the actual data taus for intersections.
requirement_at_data = 1e-9 / taus / np.sqrt(2)
delta_from_requirement = frac_std - requirement_at_data

# Find where the measured frac_std crosses the requirement curve. This uses a
# simple linear interpolation of the difference between adjacent octave points.
intersection_taus = []
intersection_values = []
for i in range(len(taus) - 1):
    if delta_from_requirement[i] == 0:
        tau_intersection = taus[i]
    elif delta_from_requirement[i] * delta_from_requirement[i + 1] < 0:
        tau_intersection = taus[i] - delta_from_requirement[i] * (taus[i + 1] - taus[i]) / (delta_from_requirement[i + 1] - delta_from_requirement[i])
    else:
        continue

    intersection_taus.append(tau_intersection)
    intersection_values.append(1e-9 / tau_intersection / np.sqrt(2))

if delta_from_requirement[-1] == 0:
    intersection_taus.append(taus[-1])
    intersection_values.append(requirement_at_data[-1])

if intersection_taus:
    for tau_intersection, value_intersection in zip(intersection_taus, intersection_values):
        print(f'Intersection: τ = {tau_intersection:.3g}s, fractional deviation = {value_intersection:.3e}')
else:
    print('No intersection found between the data points and the requirement line.')

fig, ax = plt.subplots(figsize=(8, 6))

ax.loglog(taus, frac_std, 'o-', label=r'std$[(x(t+\tau)-x(t))/\tau]$', zorder=3)
#ax.loglog(taus, frac_rmse, 's--', label=r'RMSE$[(x(t+\tau)-x(t))/\tau]$')

ax.loglog(taus_req, requirement, color = 'black', linestyle=':', linewidth = 4, label=r'$\mathbf{1\,ns}/\mathbf{(}\sqrt{\mathbf{2}}\,\tau\mathbf{)}$ requirement', zorder=3)
# Preserve the auto-scaled y-limits before adding filled regions, otherwise the
# fills can change the apparent plot range.
y_min, y_max = ax.get_ylim()
ax.axvline(10**0, color='red', linestyle='--', linewidth=2)
ax.set_ylim(y_min, y_max)
ax.set_xlim(taus_req[0], taus_req[-1])
ax.fill_between(taus_req, y_min, requirement, color='green', alpha=0.3, zorder=0)
ax.fill_between(taus_req, requirement, y_max, color='red', alpha=0.3, zorder=0)
# Annotate and mark the first crossing if one was found.
if intersection_taus:
    ax.text(
        0.04,
        0.05,
        f'Intersection\nτ = {intersection_taus[0]:.3f} s\n$f_{{dev}}$ = {intersection_values[0]:.3e}',
        transform=ax.transAxes,
        ha='left',
        va='bottom',
        fontsize=12,
        fontweight='bold',
        bbox={'facecolor': 'white', 'edgecolor': 'black', 'alpha': 0.85},
        zorder=4,
    )

    ax.plot(
        intersection_taus[0],
        intersection_values[0],
        marker='o',
        markersize=18,
        markerfacecolor='none',
        markeredgecolor='purple',
        markeredgewidth=4,
        linestyle='None',
        zorder=4,
    )

ax.set_xlabel('Time interval τ (s)', labelpad=20, fontdict=label_font)
ax.set_ylabel('Fractional deviation', fontdict=label_font)
ax.grid(True, which='both', linestyle='--', alpha=0.5)
plt.rcParams["mathtext.default"] = "bf"
ax.legend(prop=legend_font)

for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
    tick_label.set_fontweight('bold')

fig.subplots_adjust(left=0.16, right=0.96, bottom=0.18, top=0.92)

In [ ]:
# This recomputes the de-trended MTIE. This can be very slow depending
# on the amount of data and it doesn't cache data so it's commented out by default. 
# You can run it if you want to see how the MTIE changes after removing the linear,
# trend but it may take a while to complete.
if False:
	plt.close('all')
	# Formatting for plots
	label_font = {"fontname": "sans-serif", "size": "16", "color": "black", "weight": "bold", "verticalalignment": "bottom"}
	title_font = {"fontname": "sans-serif", "size": "16", "color": "black", "weight": "bold"}
	legend_font = {"family": "sans-serif", "size": "12", "style": "normal", "weight": "bold"}

	elapsed_time = np.arange(len(unwrapped_phase)) / rate
	tie_ns = (unwrapped_phase - unwrapped_phase[0]) * 1e9
	tie_slope_ns_per_s, tie_intercept_ns = np.polyfit(elapsed_time, tie_ns, 1)
	tie_fit_ns = tie_slope_ns_per_s * elapsed_time + tie_intercept_ns
	tie_residual_ns = tie_ns - tie_fit_ns
	detrended_phase = tie_residual_ns * 1e-9

	taus_detrended, mtie_detrended, mtie_err_detrended, ns_detrended = allantools.mtie(
		detrended_phase,
		rate=rate,
		data_type='phase',
		taus='octave'
	)

	for val in zip(taus_detrended, mtie_detrended):
		print(f'{val[0]:.3e} s: {val[1]*1e9:.3f} ns')

		fig, ax = plt.subplots(figsize=(8, 6))
		ax.errorbar(taus_detrended, mtie_detrended * 1e9, yerr=mtie_err_detrended * 1e9,
					fmt='o-', capsize=3, label='Detrended tinypfa LB test', alpha=0.8)

		ax.set_xscale('log')
		ax.set_xlim(taus_detrended[0] / 1.3, taus_detrended[-1] * 1.5)
		ax.set_title('Detrended MTIE', fontdict=title_font)
		ax.set_xlabel('Time Interval τ (s)', labelpad=15, fontdict=label_font)
		ax.set_ylabel('Detrended MTIE (ns)', fontdict=label_font)
		ax.tick_params(axis='both', labelsize=12)
		for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
			tick_label.set_fontweight('bold')
		ax.grid(True, which='both', linestyle='--', alpha=0.5)
		# ax.legend(prop=legend_font)
		ax.margins(y=0.1)
		fig.subplots_adjust(left=0.18, right=0.96, bottom=0.18, top=0.92)

In [ ]:
# Quick sanity check on the raw phase series before trimming/unwrapping effects.
# Large steps near +/-100 ns would suggest wrapped data or discontinuities.
phase_ns = phase * 1e9
dphase_ns = np.diff(phase_ns)

print(f"min phase: {phase_ns.min():.3f} ns")
print(f"max phase: {phase_ns.max():.3f} ns")
print(f"range: {np.ptp(phase_ns):.3f} ns")
print(f"min step: {dphase_ns.min():.3f} ns")
print(f"max step: {dphase_ns.max():.3f} ns")
print(f"steps > 50 ns: {np.sum(dphase_ns > 50)}")
print(f"steps < -50 ns: {np.sum(dphase_ns < -50)}")
print(f"steps > 90 ns: {np.sum(dphase_ns > 90)}")
print(f"steps < -90 ns: {np.sum(dphase_ns < -90)}")


In [ ]:
plt.close('all')

# Compute histogram data for 1-second averaged phase differences.
# This summarizes short-term step-to-step behavior in ps; it is not a direct
# statistical test of whether the fitted TIE slope is zero.
averaged_diffs = np.diff(unwrapped_phase[:int((len(unwrapped_phase) // rate) * rate)].reshape(-1, int(rate)).mean(axis=1))
data_ns = averaged_diffs * 1e9

data_ps = data_ns * 1000
hist_mean_ps = np.mean(data_ps)
hist_std_ps = np.std(data_ps, ddof=1)
hist_min_ps = np.min(data_ps)
hist_max_ps = np.max(data_ps)

print(f'Histogram sample count: {len(data_ps)}')
print(f'Mean: {hist_mean_ps:.3f} ps')
print(f'Std dev: {hist_std_ps:.3f} ps')
print(f'Min: {hist_min_ps:.3f} ps')
print(f'Max: {hist_max_ps:.3f} ps')

label_font = {"fontname": "sans-serif", "size": 16, "color": "black", "weight": "bold"}
legend_font = {"family": "sans-serif", "size": 12, "style": "normal", "weight": "bold"}

title_font = {"fontname": "sans-serif", "size": 16, "color": "black", "weight": "bold"}

from scipy.stats import chi2 as chi2_dist

bins = 60
total_counts = len(data_ps)
counts, edges = np.histogram(data_ps, bins=bins)
bin_centers = (edges[:-1] + edges[1:]) / 2
bin_width = edges[1] - edges[0]

pdf_norm = 1.0 / (hist_std_ps * np.sqrt(2 * np.pi))
expected_counts = total_counts * pdf_norm * np.exp(-0.5 * ((bin_centers - hist_mean_ps) / hist_std_ps) ** 2) * bin_width
chi2 = np.sum((counts - expected_counts) ** 2 / np.where(expected_counts > 0, expected_counts, 1.0))
dof = len(counts) - 3
chi2_dof = chi2 / dof
p_value = chi2_dist.sf(chi2, dof)

fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(data_ps, bins=edges, alpha=0.8, edgecolor='black', label='Data')

x_fit = np.linspace(edges[0], edges[-1], 400)
y_fit = total_counts * pdf_norm * np.exp(-0.5 * ((x_fit - hist_mean_ps) / hist_std_ps) ** 2) * bin_width
ax.plot(x_fit, y_fit, color='red', linewidth=2.0, label='Gaussian fit')

ax.set_title('Histogram of 1-second averaged phase differences', fontdict=title_font)
ax.set_xlabel('Difference (ps)', fontdict=label_font)
ax.set_ylabel('Counts', fontdict=label_font)
ax.set_yscale('symlog', linthresh=1)
ax.set_ylim(bottom=0)
ax.tick_params(axis='both', labelsize=12)
for tick_label in ax.get_xticklabels() + ax.get_yticklabels():
    tick_label.set_fontweight('bold')

fit_text = (
    f'μ = {hist_mean_ps:.3f} ps\n'
    f'σ = {hist_std_ps:.3f} ps\n'
    f'χ²/DOF = {chi2_dof:.3f}\n'
    f'p = {p_value:.3e}'
)
ax.text(
    0.98,
    0.95,
    fit_text,
    transform=ax.transAxes,
    ha='right',
    va='top',
    fontsize=12,
    bbox={'facecolor': 'white', 'edgecolor': 'black', 'alpha': 0.8}
)
ax.legend(prop=legend_font, loc='upper left')
ax.grid(True, which='both', linestyle='--', alpha=0.5)
fig.subplots_adjust(left=0.16, right=0.96, bottom=0.18, top=0.92)
